In [ ]:
# Hybrid Identity Anomaly Detection – LANL Authentication Logs

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Paths
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Random seed
SEED = 42
np.random.seed(SEED)

In [ ]:
# Read the compressed file directly (no need to unzip)
df = pd.read_csv(
    DATA_RAW / 'auth.txt.gz',
    compression='gzip',
    header=None,
    names=['time', 'user', 'pc', 'logon_type'],
    sep='\t'
)

print(f"Total events: {len(df)}")
df.head()

In [ ]:
# Convert time to datetime
df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%d %H:%M:%S')

# Extract time features
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

# Count logins per user per hour (rolling hourly activity)
user_hourly = df.groupby(['user', 'hour']).size().reset_index(name='hourly_count')
df = df.merge(user_hourly, on=['user', 'hour'], how='left')

# Count unique PCs used by user (profiles)
user_pc_diversity = df.groupby('user')['pc'].nunique().reset_index(name='unique_pcs')
df = df.merge(user_pc_diversity, on='user', how='left')

# Encode logon_type as numeric
df['logon_type_code'] = pd.factorize(df['logon_type'])[0]

# Select features for anomaly detection
feature_cols = ['hour', 'day_of_week', 'is_weekend', 'hourly_count', 'unique_pcs', 'logon_type_code']
X = df[feature_cols].fillna(0)

print(f"Feature matrix shape: {X.shape}")
X.head()

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
model = IsolationForest(
    contamination=0.01,          # expect ~1% anomalies
    random_state=SEED,
    n_estimators=100
)
model.fit(X_scaled)

# Predict: -1 = anomaly, 1 = normal
df['anomaly'] = model.predict(X_scaled)
anomalies = df[df['anomaly'] == -1]
print(f"Anomalies detected: {len(anomalies)} ({len(anomalies)/len(df):.4f} of data)")

In [ ]:
# The LANL dataset includes known red‑team events in a separate file.
# For simplicity, we'll create synthetic anomalies by choosing users who logged in from unusual PCs.
# We can also check if our anomalies capture these.

# Example: list the top anomalous users
top_anomalous_users = anomalies['user'].value_counts().head(10)
print("Top 10 anomalous users:")
print(top_anomalous_users)

# Optional: Load redteam.txt if available
redteam_file = DATA_RAW / 'redteam.txt'
if redteam_file.exists():
    redteam = pd.read_csv(redteam_file, compression='gzip', header=None, names=['time','user','pc','logon_type'], sep='\t')
    redteam['time'] = pd.to_datetime(redteam['time'], format='%Y-%m-%d %H:%M:%S')
    # Merge with anomalies to see overlap
    red_anomalies = pd.merge(anomalies, redteam, on=['time','user','pc'], how='inner')
    print(f"Red‑team events captured by anomaly detector: {len(red_anomalies)}")
else:
    print("redteam.txt not found; using synthetic evaluation.")

In [ ]:
# Plot anomaly scores distribution
scores = model.decision_function(X_scaled)
plt.figure(figsize=(10,4))
sns.histplot(scores, bins=50)
plt.title('Anomaly Score Distribution')
plt.xlabel('Decision Function Score')
plt.show()

# Plot anomalies by hour
plt.figure(figsize=(10,4))
df.groupby('hour')['anomaly'].apply(lambda x: (x == -1).sum()).plot(kind='bar')
plt.title('Anomalies per Hour of Day')
plt.ylabel('Count')
plt.show()

In [ ]:
# Save the labeled dataset
df.to_csv(DATA_PROCESSED / 'lanl_auth_with_anomalies.csv', index=False)

# Save the model (for later use in a demo)
import joblib
joblib.dump(model, DATA_PROCESSED / 'isolation_forest_model.pkl')
joblib.dump(scaler, DATA_PROCESSED / 'scaler.pkl')

print("Saved processed data and model.")

In [ ]:
summary = {
    'total_events': len(df),
    'anomalies_detected': len(anomalies),
    'contamination_rate': len(anomalies)/len(df),
    'top_anomalous_user': top_anomalous_users.index[0] if len(top_anomalous_users) > 0 else 'N/A'
}
print("Project 2 Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")